# SFT Stage-7 — run on the A100 (open THIS notebook in Cursor)

Connect the kernel to your **A100** (kernel selector → Colab → **Auto Connect**), then run top to bottom. **CELL 0 self-provisions** the runtime (clone/install/token/stage), so a fresh or recycled VM works — no separate readiness notebook needed.

The `sft-improve` goal / `bilevel-colab-loop` skill rewrites the tagged candidate cell and Computer Use presses **Run All**; each `METRIC=` line lands in this local file for Codex to read.

## CELL 0 — provision the runtime (idempotent; run first)
Works on a **fresh OR warm** A100. Clones the repo, installs deps, loads `HF_TOKEN`, and stages the corpus **only if missing**. ⚠ On a brand-new VM this does the ~9.85 GB staging pull (a few minutes). If your earlier runtime is still alive, prefer reconnecting to it (**Auto Connect**) so this is a no-op.

In [ ]:
import os, pathlib, subprocess, sys

REPO = '/content/SLM'
# 1) clone if missing (idempotent), then checkout main + pull
if not pathlib.Path(REPO, '.git').is_dir():
    print('cloning -> /content/SLM ...')
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'checkout', 'main'], check=True)
subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
print('HEAD:', subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout.strip())

# 2) training deps (Colab ships torch/transformers; adds peft/trl/bitsandbytes/accelerate/qwen-vl-utils)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft]'], check=True)

# 3) HF token for the PRIVATE dataset + Qwen download. Prefer an uploaded .env, else paste (read scope OK).
def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                s = _l.strip()
                if s.startswith(name + '='):
                    return s.split('=', 1)[1].strip().strip('"').strip("'")
    return None
if not os.environ.get('HF_TOKEN'):
    tok = _env_token('HF_TOKEN')
    if not tok:
        import getpass
        tok = getpass.getpass('Paste HF_TOKEN (read scope OK for staging): ').strip()
    os.environ['HF_TOKEN'] = tok
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))

# 4) stage the corpus ONLY if missing  (weights gitignored -> come only from staging; ~9.85GB)
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('/content/slm/tokenizer/final/model.pt').is_file():
    print('corpus missing -> staging ~9.85GB from hf://datasets/ericrcwu/LUT_SLM ...')
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
else:
    print('corpus already staged at /content/slm')
print('contents:', sorted(os.listdir('/content/slm')) if pathlib.Path('/content/slm').is_dir() else 'MISSING')


In [ ]:
import torch, transformers
import bitsandbytes as bnb, peft, accelerate, qwen_vl_utils  # must all import
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor  # VL class must exist
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
print('transformers', transformers.__version__, 'bnb', bnb.__version__,
      'peft', peft.__version__, 'accelerate', accelerate.__version__)
assert torch.cuda.is_available(), 'no CUDA — wrong runtime (need A100)'
print('imports OK')


In [ ]:
import os, pathlib, shutil
os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'   # lowercase corpus (images + tokenizer)
repo, corpus = pathlib.Path('/content/SLM'), pathlib.Path('/content/slm')
assert repo.is_dir(), 'repo clone missing at /content/SLM'
assert corpus.is_dir(), 'staged corpus missing at /content/slm (run readiness stage)'
tok = corpus / 'tokenizer' / 'final' / 'model.pt'
assert tok.is_file(), f'frozen tokenizer weights missing: {tok}'
free_gb = shutil.disk_usage('/content').free / 1e9
print('SLM_ARTIFACT_ROOT =', os.environ['SLM_ARTIFACT_ROOT'], '| free /content GB:', round(free_gb,1))
assert free_gb > 25, 'need >25GB free for the fp32 base_resized (~12-14GB) + caches'
print('pre-run asserts OK')


## CELL 3 — vocab resize + preflight  🛑 expect `[vocab-resize][OK]` + identity bound
Downloads Qwen2.5-VL-3B (~6 GB) and writes a full fp32 `models/base_resized` (~12-14 GB). Run it **once** (unseeded new-token init — re-running trains a different base).

In [ ]:
# cheap dry run first: proves the 4-gate preflight + model-class import + tied status, writes nothing
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.vocab_resize --preflight-only


In [ ]:
import json, pathlib
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.vocab_resize --out models/base_resized
d = pathlib.Path('/content/SLM/models/base_resized')
assert (d/'preprocessor_config.json').is_file(), 'base_resized missing preprocessor_config.json (AutoProcessor would fail)'
m = json.load(open(d/'vocab_resize_manifest.json'))
EXPECT = 'vq_v2_srgbres_17to4_cb256_t64__w91cffdd2c82f'
assert m.get('tokenizer_version') == EXPECT, ('tokenizer_version mismatch/null', m.get('tokenizer_version'))
assert m.get('vq_codebook_sha256'), 'null vq_codebook_sha256 — identity did not bind (check SLM_ARTIFACT_ROOT)'
print('vocab_resize OK; identity bound:', m['tokenizer_version'])


## GATE 0 — one OBSERVABLE smoke run (must pass before the loop)  🛑
Uses `gradient_accumulation_steps=4`/`effective_batch_size=4` so ≥10 optimizer steps land past warmup and loss prints. Confirm: `steps>0`, ≥2 falling loss lines, `[sft][OK]`, and a finite `METRIC=` from the scorer. If `[sft][skip]` approaches the row count → images aren't resolving (case trap) → STOP.

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/sft_default.yaml'))
cfg.update(gradient_accumulation_steps=4, effective_batch_size=4)  # observability on smoke-50
yaml.safe_dump(cfg, open('gate0.yaml','w'))
print('gate0.yaml triple:', {k: cfg[k] for k in ('per_device_batch_size','gradient_accumulation_steps','effective_batch_size')})


In [ ]:
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.train --config gate0.yaml \
    --resized-model models/base_resized --smoke-size 50 --run-id gate0
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.score_tokens --config gate0.yaml \
    --resized-model models/base_resized --adapter models/sft_adapters/gate0_smoke50 --limit 32


## BILEVEL LOOP — candidate eval (driven by the `bilevel-colab-loop` skill)
The skill rewrites the tagged candidate cell below; Computer Use runs **Run All**; the bridge trains + scores and prints one `METRIC=` line that Codex reads from this file.

In [ ]:
# HF UPLOAD SETUP — run once. Adapters are uploaded to PUSH_HF_REPO after each eval.
# Leave PUSH_HF_REPO empty to skip upload. Upload needs a WRITE token (read-only HF_TOKEN 403s).
import os, getpass, pathlib
os.environ['PUSH_HF_REPO'] = 'ericrcwu/LUT_SLM_sft_adapters'   # <-- your HF model repo (created if missing)
if not os.environ.get('HF_WRITE_TOKEN'):
    for _p in ('/content/SLM/.env', '/content/.env'):
        if pathlib.Path(_p).exists():
            for _l in pathlib.Path(_p).read_text().splitlines():
                if _l.strip().startswith('HF_WRITE_TOKEN='):
                    os.environ['HF_WRITE_TOKEN'] = _l.split('=',1)[1].strip().strip('"').strip("'")
if os.environ['PUSH_HF_REPO'] and not os.environ.get('HF_WRITE_TOKEN'):
    os.environ['HF_WRITE_TOKEN'] = getpass.getpass('Paste HF WRITE token (SLM_Alpha_Write): ').strip()
print('PUSH_HF_REPO =', os.environ.get('PUSH_HF_REPO'), '| write token set:', bool(os.environ.get('HF_WRITE_TOKEN')))


In [ ]:
# @candidate-config  (auto-written by bilevel-colab-loop; do not hand-edit)
import json, pathlib
_CANDIDATE = json.dumps({
    'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'learning_rate_lora': 0.0002, 'warmup_ratio': 0.03,
    'max_grad_norm': 1.0, 'weight_decay': 0.0, 'max_pixels': 200704,
}, indent=2)
pathlib.Path('/content/SLM/candidate.json').write_text(_CANDIDATE)
print('candidate.json written:', _CANDIDATE)


In [ ]:
!SLM_ARTIFACT_ROOT=/content/slm python -m sft.bilevel_bridge --mode colab \
    --config /content/SLM/candidate.json --resized-model models/base_resized \
    --smoke-size 200 --score-limit 48 --run-id bl
